# Telecom Customer Churn Prediction

**Halimata Ndiaye – ENSIIE | December 2025**

---

This notebook presents a complete machine learning pipeline to predict customer churn for a telecom operator. It covers exploratory data analysis, preprocessing, supervised classification, hyperparameter tuning, and a fairness analysis by gender.

> 📄 A detailed report (in French) is available in the `report/` folder.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 2. Load Data

In [ ]:
df = pd.read_csv("../data/celldata.csv")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Exploratory Data Analysis

### 3.1 Dataset Overview

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

### 3.2 Missing Values

In [ ]:
missing = df.isna().sum()
print(missing)
print("\nNo missing values." if missing.sum() == 0 else f"Total missing values: {missing.sum()}")

### 3.3 Target Variable Distribution (Churn)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Churn', palette='viridis')
plt.title('Target Variable Distribution – Churn', fontsize=14, fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
for i, count in enumerate(df['Churn'].value_counts().sort_index()):
    plt.text(i, count + 50, f'{count}\n({count/len(df):.1%})',
             ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()
print("Slight class imbalance detected: F1-score will be used as the main selection criterion.")

### 3.4 Continuous Variables Distribution by Churn Status

In [ ]:
num_vars = ['CreditScore', 'Age', 'Balance', 'Salary', 'Tenure']

for var in num_vars:
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x=var, hue='Churn', kde=True, palette='viridis', alpha=0.7)
    plt.title(f'Distribution of {var} by Churn Status', fontsize=14, fontweight='bold')
    plt.xlabel(var, fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

Distributions largely overlap for most continuous variables. **Age** is the most discriminating feature: churners tend to be older on average. No single variable clearly separates the two groups, which justifies combining multiple features in the models.

### 3.5 Correlation Matrix

In [ ]:
correlation_matrix = df[['Churn', 'Age', 'Balance', 'CreditScore']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_matrix, annot=True, cmap='coolwarm',
    center=0, fmt='.2f', square=True, linewidths=1
)
plt.title('Correlation Matrix – Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Age shows the highest positive correlation with churn (0.27). Other variables show weaker correlations, confirming the need to combine several features in the models.

### 3.6 Churn Rate by Categorical Variable

In [ ]:
for gender in ['Male', 'Female']:
    gender_df = df[df['Gender'] == gender]
    plt.figure(figsize=(8, 5))
    sns.countplot(data=gender_df, x='Churn', palette='viridis')
    plt.title(f'Churn – {gender}s', fontsize=14, fontweight='bold')
    plt.xlabel('Churn Status')
    plt.ylabel(f'Number of {gender} Customers')
    plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
    total = len(gender_df)
    for container in plt.gca().containers:
        for bar in container:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., h + 5,
                     f'{int(h)}\n({h/total:.1%})', ha='center', va='bottom', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
for n in [1, 2, 3]:
    nb_df = df[df['NumOfProducts'] == n]
    plt.figure(figsize=(8, 5))
    sns.countplot(data=nb_df, x='Churn', palette='viridis')
    plt.title(f'Churn – {n} Product(s)', fontsize=14, fontweight='bold')
    plt.xlabel('Churn Status')
    plt.ylabel(f'Number of Customers ({n} product(s))')
    plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
    total = len(nb_df)
    for container in plt.gca().containers:
        for bar in container:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., h + 5,
                     f'{int(h)}\n({h/total:.1%})', ha='center', va='bottom', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
for country in ['Germany', 'France', 'Spain']:
    country_df = df[df['Geography'] == country]
    plt.figure(figsize=(8, 5))
    sns.countplot(data=country_df, x='Churn', palette='viridis')
    plt.title(f'Churn – {country}', fontsize=14, fontweight='bold')
    plt.xlabel('Churn Status')
    plt.ylabel(f'Number of Customers in {country}')
    plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
    total = len(country_df)
    for container in plt.gca().containers:
        for bar in container:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., h + 5,
                     f'{int(h)}\n({h/total:.1%})', ha='center', va='bottom', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
for m in [0, 1]:
    active_df = df[df['IsActiveMember'] == m]
    label = 'Active' if m == 1 else 'Inactive'
    plt.figure(figsize=(8, 5))
    sns.countplot(data=active_df, x='Churn', palette='viridis')
    plt.title(f'Churn – {label} Members', fontsize=14, fontweight='bold')
    plt.xlabel('Churn Status')
    plt.ylabel(f'Number of {label} Members')
    plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
    total = len(active_df)
    for container in plt.gca().containers:
        for bar in container:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., h + 5,
                     f'{int(h)}\n({h/total:.1%})', ha='center', va='bottom', fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

**Key findings:**
- **Germany** has a churn rate of ~31%, significantly higher than France and Spain (~16.5%)
- **Female** customers churn ~12% more than male customers
- Customers with **3 products** have a churn rate of ~83% — a very strong signal
- **Inactive members** churn significantly more than active ones

## 4. Data Preprocessing

No missing values or inconsistent entries were detected during EDA. The following transformations are applied:

In [ ]:
# Copy of the original DataFrame
df_enc = df.copy()

# Binary encoding of Gender (Male=1, Female=0)
df_enc["Gender"] = df_enc["Gender"].map({"Male": 1, "Female": 0})

# One-hot encoding of Geography (drop_first to avoid multicollinearity)
df_enc = pd.get_dummies(df_enc, columns=["Geography"], drop_first=True)

# Features / target split
y = df_enc["Churn"]
X = df_enc.drop(columns=["Churn"])

# Stratified train (70%) / test (30%) split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

# Standardization for scale-sensitive models (LogReg, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")
print(f"Features: {list(X.columns)}")

## 5. Anomaly Detection (IsolationForest)

We test the impact of cleaning the training data with IsolationForest before fitting the classifiers.

In [ ]:
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=0)
iso.fit(X_train)
y_pred_out = iso.predict(X_train)  # +1 = inlier, -1 = outlier

mask_inliers  = (y_pred_out == 1)
X_train_if    = X_train[mask_inliers]
y_train_if    = y_train[mask_inliers]
X_train_if_scaled = scaler.fit_transform(X_train_if)

print(f"Train size before filtering : {X_train.shape[0]}")
print(f"Train size after filtering  : {X_train_if.shape[0]}")
print(f"Outliers removed            : {(~mask_inliers).sum()}")

## 6. Modelling

### 6.1 Evaluation Function

In [ ]:
def eval_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train a model and return train/test metrics."""
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    if hasattr(model, "predict_proba"):
        y_proba_te = model.predict_proba(X_te)[:, 1]
        auc_te = roc_auc_score(y_te, y_proba_te)
    else:
        y_proba_te = None
        auc_te = np.nan

    metrics = {
        "model":       name,
        "acc_train":   accuracy_score(y_tr, y_pred_tr),
        "f1_train":    f1_score(y_tr, y_pred_tr),
        "acc_test":    accuracy_score(y_te, y_pred_te),
        "prec_test":   precision_score(y_te, y_pred_te),
        "recall_test": recall_score(y_te, y_pred_te),
        "f1_test":     f1_score(y_te, y_pred_te),
        "auc_test":    auc_te,
    }
    return metrics, model, y_pred_te, y_proba_te

### 6.2 Candidate Models

Five models are compared, covering a spectrum from interpretability (LogReg, Tree) to predictive performance (RF, GB). Since the dataset is imbalanced, **F1-score** is used as the main selection criterion.

In [ ]:
models = [
    ("LogReg", LogisticRegression(max_iter=500, class_weight="balanced")),
    ("KNN",    KNeighborsClassifier(n_neighbors=5)),
    ("Tree",   DecisionTreeClassifier(max_depth=5, random_state=0)),
    ("RF",     RandomForestClassifier(n_estimators=200, random_state=0, class_weight="balanced")),
    ("GB",     GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=0)),
]

### 6.3 Results Without IsolationForest Filtering

In [ ]:
results = []
for name, clf in models:
    Xtr = X_train_scaled if name in ["LogReg", "KNN"] else X_train
    Xte = X_test_scaled  if name in ["LogReg", "KNN"] else X_test
    m, _, _, _ = eval_model(name, clf, Xtr, y_train, Xte, y_test)
    m["scenario"] = "without_if"
    results.append(m)

df_res_no_if = pd.DataFrame(results)
df_res_no_if

### 6.4 Results With IsolationForest Filtering

In [ ]:
results_if = []
for name, clf in models:
    Xtr = X_train_if_scaled if name in ["LogReg", "KNN"] else X_train_if
    Xte = X_test_scaled     if name in ["LogReg", "KNN"] else X_test
    m, _, _, _ = eval_model(name, clf, Xtr, y_train_if, Xte, y_test)
    m["scenario"] = "with_if"
    results_if.append(m)

df_res_if = pd.DataFrame(results_if)
df_res_if

### 6.5 Consolidated Results (sorted by F1 and AUC)

In [ ]:
df_results = pd.concat([df_res_no_if, df_res_if], ignore_index=True)
df_results.sort_values(by=["f1_test", "auc_test"], ascending=False)

**Gradient Boosting without IsolationForest** achieves the best F1-score. The **Decision Tree** is retained alongside it for its interpretability: it produces explicit decision rules that can be directly communicated to marketing teams.

## 7. Hyperparameter Tuning

Grid search with 5-fold cross-validation, optimizing F1-score.

In [ ]:
param_grid_tree = {"max_depth": [3, 5, 7, 9, None]}
tree_cv   = DecisionTreeClassifier(random_state=0)
grid_tree = GridSearchCV(tree_cv, param_grid_tree, scoring="f1", cv=5, n_jobs=-1)
grid_tree.fit(X_train, y_train)
print("Best parameters (Tree):", grid_tree.best_params_)
print("Best F1 (CV)           :", round(grid_tree.best_score_, 4))

In [ ]:
param_grid_gb = {
    "n_estimators":  [100, 200],
    "learning_rate": [0.05, 0.1],
}
gb_cv   = GradientBoostingClassifier(random_state=0)
grid_gb = GridSearchCV(gb_cv, param_grid_gb, scoring="f1", cv=5, n_jobs=-1)
grid_gb.fit(X_train, y_train)
print("Best parameters (GB):", grid_gb.best_params_)
print("Best F1 (CV)        :", round(grid_gb.best_score_, 4))

## 8. Optimised Model Evaluation

In [ ]:
# Retrain with best hyperparameters
tree = DecisionTreeClassifier(
    max_depth=grid_tree.best_params_["max_depth"], random_state=0
)
tree.fit(X_train, y_train)
y_pred_tree  = tree.predict(X_test)
y_proba_tree = tree.predict_proba(X_test)[:, 1]

gb = GradientBoostingClassifier(
    n_estimators=grid_gb.best_params_["n_estimators"],
    learning_rate=grid_gb.best_params_["learning_rate"],
    random_state=0
)
gb.fit(X_train, y_train)
y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

def print_eval(name, y_true, y_pred, y_proba):
    print(f"{'='*40}\n{name}\n{'='*40}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print(classification_report(y_true, y_pred, digits=3))
    print(f"AUC: {roc_auc_score(y_true, y_proba):.4f}\n")

print_eval("Decision Tree", y_test, y_pred_tree, y_proba_tree)
print_eval("Gradient Boosting", y_test, y_pred_gb, y_proba_gb)

### 8.1 ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_proba_tree, name="Decision Tree", ax=axes[0])
axes[0].set_title("ROC Curve – Decision Tree", fontsize=13, fontweight='bold')
axes[0].plot([0, 1], [0, 1], '--', color='grey')

RocCurveDisplay.from_predictions(y_test, y_proba_gb, name="Gradient Boosting", ax=axes[1])
axes[1].set_title("ROC Curve – Gradient Boosting", fontsize=13, fontweight='bold')
axes[1].plot([0, 1], [0, 1], '--', color='grey')

plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

importances_tree = pd.Series(tree.feature_importances_, index=X_train.columns)
importances_tree.sort_values(ascending=False).plot(kind="bar", ax=axes[0], color='steelblue')
axes[0].set_title("Feature Importance – Decision Tree", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Importance")
axes[0].tick_params(axis='x', rotation=45)

importances_gb = pd.Series(gb.feature_importances_, index=X_train.columns)
importances_gb.sort_values(ascending=False).plot(kind="bar", ax=axes[1], color='darkorange')
axes[1].set_title("Feature Importance – Gradient Boosting", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Importance")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In both models, **Age** and **NumOfProducts** are the most influential features. Gradient Boosting spreads importance more broadly across variables (Balance, IsActiveMember, CreditScore, Geography_Germany), reflecting its ability to capture complex interactions between features.

## 10. Fairness Analysis – Sensitive Attribute: Gender

We evaluate model fairness along three standard criteria: **Independence**, **Separation**, and **Sufficiency**.

In [ ]:
# Retrieve original gender labels for the test set
gender_test = df.loc[X_test.index, "Gender"]  # "Male" / "Female"

res_tree = pd.DataFrame({
    "y_true":  y_test.values,
    "y_pred":  y_pred_tree,
    "y_score": y_proba_tree,
    "Gender":  gender_test.values
})

res_gb = pd.DataFrame({
    "y_true":  y_test.values,
    "y_pred":  y_pred_gb,
    "y_score": y_proba_gb,
    "Gender":  gender_test.values
})

### 10.1 Independence (Demographic Parity)

Proportion of customers predicted as churners by gender: P(ŷ=1 | Gender)

In [ ]:
def independence(df_res, model_name):
    print(f"=== {model_name} – Independence ===")
    for g in ["Male", "Female"]:
        p_hat = df_res.loc[df_res["Gender"] == g, "y_pred"].mean()
        print(f"  {g}: P(ŷ=1) = {p_hat:.3f}")
    print()

independence(res_tree, "Decision Tree")
independence(res_gb, "Gradient Boosting")

### 10.2 Separation (TPR and FPR by Gender)

True positive rate and false positive rate conditioned on gender.

In [ ]:
def tpr_fpr_by_gender(df_res, model_name):
    print(f"=== {model_name} – Separation ===")
    for g in ["Male", "Female"]:
        sub = df_res[df_res["Gender"] == g]
        cm  = confusion_matrix(sub["y_true"], sub["y_pred"])
        tn, fp, fn, tp = cm.ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        print(f"  {g}: TPR={tpr:.3f}, FPR={fpr:.3f}")
    print()

tpr_fpr_by_gender(res_tree, "Decision Tree")
tpr_fpr_by_gender(res_gb, "Gradient Boosting")

### 10.3 Sufficiency (Conditional Calibration)

For a given predicted score, is the actual churn rate consistent across genders?

In [ ]:
def calibration_by_gender(df_res, model_name, n_bins=5):
    print(f"=== {model_name} – Sufficiency ===")
    for g in ["Male", "Female"]:
        sub = df_res[df_res["Gender"] == g].copy()
        sub["bin"] = pd.qcut(sub["y_score"], q=n_bins, duplicates="drop")
        calib = sub.groupby("bin", observed=True).agg(
            mean_score=("y_score", "mean"),
            frac_churn=("y_true",  "mean"),
            count      =("y_true",  "size")
        )
        print(f"\n  Gender = {g}")
        print(calib.to_string())
    print()

calibration_by_gender(res_tree, "Decision Tree")
calibration_by_gender(res_gb,   "Gradient Boosting")

## 11. Conclusion & Business Recommendations

### Model Performance Summary

| Model | F1 (test) | AUC (test) | Interpretability |
|---|---|---|---|
| **Gradient Boosting** | **~0.58** | **0.86** | Low |
| Decision Tree | ~0.55 | 0.82 | **High** |

### Key Churn Drivers
- **Age**: older customers are more likely to churn
- **NumOfProducts**: 3 products → churn rate ~83%
- **Geography**: Germany shows a significantly higher churn rate (~31%)
- **IsActiveMember**: inactive customers churn at a much higher rate

### Fairness Summary
Both models generate more churn alerts for female customers (GB: ~18.5%) than for male customers (~9.8%), partially reflecting a real pattern in the data. TPR/FPR gaps remain moderate — no major bias detected.

### Business Recommendations
- Target **older customers** and **German customers** with dedicated retention offers
- Flag customers with **3 products** as high-risk and intervene proactively
- Re-engage **inactive members** with personalised incentives
- Use the Decision Tree's explicit rules to communicate churn risk factors to marketing teams